##data Injestion

In [8]:
##Datastructure
from langchain_core.documents import Document

In [9]:
doc=Document(
    page_content="this is the main text used to create RAG",
    metadata={
        "source": "example.txt",
        "pages": 217,
        "author_created": "Krish Naik",
        "date_created": "2025-06-12"

    }
)

print(doc)

page_content='this is the main text used to create RAG' metadata={'source': 'example.txt', 'pages': 217, 'author_created': 'Krish Naik', 'date_created': '2025-06-12'}


In [10]:
import os
os.makedirs("../data/text_files", exist_ok=True)


In [11]:
sample_texts={
    "../data/text_files/python_intro.txt": """Python programming Introduction

Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation,[38] an extensive ("batteries-included") standard library, and garbage collection. Python supports multiple programming paradigms but with an emphasis on object-oriented programming and dynamic typing.
Python's core philosophy is summarized in the Zen of Python (PEP 20) written by Tim Peters, which includes aphorisms such as these:Explicit is better than implicit.
Simple is better than complex.
Readability counts.
Special cases aren't special enough to break the rules.

However, Python has received criticism for violating these principles and adding unnecessary language bloat. Responses to these criticisms note that the Zen of Python is a guideline rather than a rule. The addition of some new features had been controversial: Guido van Rossum resigned as Benevolent Dictator for Life after conflict about adding the assignment expression operator in Python.""",

    "../data/text_files/machine learning_intro.txt": """machine learning basics

Machine learning is a type of artificial intelligence where computers learn from data instead of using rigid rules written by humans. The three core approaches are supervised learning, unsupervised learning, and reinforcement learning
Core Types of Machine Learning
Supervised Learning: Trains on labeled data (input and correct output provided) to predict values or categories, such as linear regression or spam filters.
Unsupervised Learning: Finds hidden patterns or groups in unlabeled data, such as customer clustering or market segmentation."""

}

for filepath, content in sample_texts.items():
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)
print("Sample text files created")

     



Sample text files created


In [12]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader(
    "../data/text_files/python_intro.txt",
    encoding="utf-8"
)

documents = loader.load()

print(documents)


C:\Users\Hp\AppData\Local\Temp\ipykernel_18864\4287664830.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python programming Introduction\n\nPython is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation,[38] an extensive ("batteries-included") standard library, and garbage collection. Python supports multiple programming paradigms but with an emphasis on object-oriented programming and dynamic typing.\nPython\'s core philosophy is summarized in the Zen of Python (PEP 20) written by Tim Peters, which includes aphorisms such as these:Explicit is better than implicit.\nSimple is better than complex.\nReadability counts.\nSpecial cases aren\'t special enough to break the rules.\n\nHowever, Python has received criticism for violating these principles and adding unnecessary language bloat. Responses to these criticisms note that the Zen of Python is a guideline rather than a rule. The addition of some new feat

In [13]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

text_loader = DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=False
)

text_documents = text_loader.load()

print(f"Loaded {len(text_documents)} text documents")

Loaded 2 text documents


In [14]:
from langchain_community.document_loaders import PyMuPDFLoader

pdf_loader = DirectoryLoader(
    "../data/pdf_files",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=False
)

pdf_documents = pdf_loader.load()

print(f"Loaded {len(pdf_documents)} PDF documents")

Loaded 97 PDF documents


In [15]:
type(pdf_documents[0])

langchain_core.documents.base.Document

In [16]:
documents = text_documents + pdf_documents

print(f"Total documents: {len(documents)}")

Total documents: 99


In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks: {len(chunks)}")

Total chunks: 157


In [18]:
import numpy as np

from sentence_transformers import SentenceTransformer

import chromadb
from chromadb.config import Settings

import uuid

from typing import List, Dict, Any, Tuple

from sklearn.metrics.pairwise import cosine_similarity

##Embedding and Vactor Store

In [18]:
from sentence_transformers import SentenceTransformer
import numpy as np
from typing import List


class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager.

        Args:
            model_name: Hugging Face model name for sentence embeddings.
        """
        self.model_name = model_name
        self.model = None
        self.__load_model()

    def __load_model(self):
        """Load the SentenceTransformer model."""
        try:
            print(f"Loading model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(
                f"Model loaded successfully. "
                f"Embedding dimension: {self.model.get_sentence_embedding_dimension()}"
            )
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.

        Args:
            texts: List of text strings.

        Returns:
            NumPy array of embeddings.
        """
        if self.model is None:
            raise ValueError("Model not loaded.")

        print(f"Generating embeddings for {len(texts)} texts...")

        embeddings = self.model.encode(
            texts,
            show_progress_bar=True
        )

        print(f"Generated embeddings with shape: {embeddings.shape}")

        return embeddings

    def get_embedding_dimension(self):
        """Return the embedding dimension."""
        if self.model is None:
            raise ValueError("Model not loaded.")

        return self.model.get_sentence_embedding_dimension()


# Initialize the manager
embedding_manager = EmbeddingManager()

c:\Users\Hp\Documents\RAG-for begineers\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6943.43it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\Hp\AppData\Local\Temp\ipykernel_5164\264766164.py:27: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  f"Embedding dimension: {self.model.get_sentence_embedding_dimension()}"


In [11]:
import os
import uuid
import chromadb
import numpy as np

from typing import List, Any

In [12]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_store"
    ):

        self.collection_name = collection_name
        self.persist_directory = persist_directory

        self.client = None
        self.collection = None

        self._initialize_store()

    def _initialize_store(self):

        os.makedirs(self.persist_directory, exist_ok=True)

        self.client = chromadb.PersistentClient(
            path=self.persist_directory
        )

        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={
                "description": "PDF document embeddings for RAG"
            }
        )

        print(f"Vector store initialized. Collection: {self.collection_name}")
        print(f"Existing documents in collection: {self.collection.count()}")

    def retrieve(self, query, embedding_manager, top_k=3):
        """Retrieve the top-k most similar documents."""

        # Generate embedding for the query
        query_embedding = embedding_manager.generate_embeddings([query])[0]

        # Query ChromaDB
        results = self.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k
        )

        retrieved_docs = []

        for doc, metadata, distance in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0]
        ):
            retrieved_docs.append({
                "content": doc,
                "metadata": metadata,
                "distance": distance
            })

        return retrieved_docs

In [13]:
vectorstore = VectorStore()

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 785


In [22]:
print(globals().keys())


dict_keys(['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh', 'In', 'Out', 'get_ipython', 'exit', 'quit', 'open', '_', '__', '___', '__vsc_ipynb_file__', '_i', '_ii', '_iii', '_i1', 'sys', 'subprocess', '_1', '_i2', 'ChatGroq', 'os', 'load_dotenv', '_i3', 'groq_api_key', '_i4', 'llM', 'rag_simple', '_i5', '_i6', '_i7', '_i8', 'Document', '_i9', 'doc', '_i10', '_i11', 'sample_texts', 'filepath', 'content', 'f', '_i12', 'TextLoader', 'loader', 'documents', '_i13', 'DirectoryLoader', 'text_loader', 'text_documents', '_i14', 'PyMuPDFLoader', 'pdf_loader', 'pdf_documents', '_i15', '_15', '_i16', '_i17', 'RecursiveCharacterTextSplitter', 'text_splitter', 'chunks', '_i18', 'np', 'SentenceTransformer', 'chromadb', 'Settings', 'uuid', 'List', 'Dict', 'Any', 'Tuple', 'cosine_similarity', '_i19', 'EmbeddingManager', 'embedding_manager', '_i20', '_i21', 'VectorStore', 'vectorstore', '_21', '_i22'])


In [28]:
%whos

Variable                         Type                              Data/Info
----------------------------------------------------------------------------
Any                              _AnyMeta                          typing.Any
Dict                             _SpecialGenericAlias              typing.Dict
DirectoryLoader                  ABCMeta                           <class 'langchain_communi<...>rectory.DirectoryLoader'>
Document                         ModelMetaclass                    <class 'langchain_core.documents.base.Document'>
EmbeddingManager                 type                              <class '__main__.EmbeddingManager'>
List                             _SpecialGenericAlias              typing.List
PyMuPDFLoader                    ABCMeta                           <class 'langchain_communi<...>aders.pdf.PyMuPDFLoader'>
RecursiveCharacterTextSplitter   ABCMeta                           <class 'langchain_text_sp<...>veCharacterTextSplitter'>
SentenceTransformer   

In [23]:
texts = [doc.page_content for doc in chunks]
embeddings = embedding_manager.generate_embeddings(texts)

vectorstore.add_documents(
    documents=chunks,
    embeddings=embeddings
)


Generating embeddings for 157 texts...


Batches: 100%|██████████| 5/5 [00:07<00:00,  1.44s/it]


Generated embeddings with shape: (157, 384)
Adding 157 documents to vector store...
Successfully added 157 documents to vector store
Total documents in collection: 471


In [25]:
### Convert text to embeddings
texts = [doc.page_content for doc in chunks]
texts


## generate embeddings
embeddings = embedding_manager.generate_embeddings(texts)

## store in the vector
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 157 texts...


Batches: 100%|██████████| 5/5 [00:06<00:00,  1.37s/it]


Generated embeddings with shape: (157, 384)
Adding 157 documents to vector store...
Successfully added 157 documents to vector store
Total documents in collection: 785


### Retriever pipeline from Vectorstore


In [26]:
from typing import List, Dict, Any


class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever.

        Args:
            vector_store: Vector store containing document embeddings.
            embedding_manager: Manager for generating query embeddings.
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, n_results: int = 3) -> Dict[str, Any]:
        """
        Retrieve the most relevant documents for a query.

        Args:
            query: User's question.
            n_results: Number of documents to retrieve.

        Returns:
            ChromaDB query results.
        """

        print(f"Searching for: {query}")

        # Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])

        # Search ChromaDB
        results = self.vector_store.collection.query(
            query_embeddings=query_embedding.tolist(),
            n_results=n_results
        )

        print(f"Retrieved {len(results['documents'][0])} documents")

        return results

    def display_results(self, results: Dict[str, Any]):
        """
        Display retrieved documents in a readable format.
        """

        documents = results["documents"][0]
        metadatas = results["metadatas"][0]
        distances = results["distances"][0]

        for i, (doc, meta, distance) in enumerate(
            zip(documents, metadatas, distances), start=1
        ):
            print("=" * 80)
            print(f"Result {i}")
            print(f"Similarity Distance: {distance:.4f}")
            print(f"Metadata: {meta}")
            print("-" * 80)
            print(doc)
            print()
            

In [27]:
retriever = RAGRetriever(
    vector_store=vectorstore,
    embedding_manager=embedding_manager
)

In [28]:
query = "What is machine learning?"

results = retriever.retrieve(
    query=query,
    n_results=3
)

Searching for: What is machine learning?
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 31.06it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents


In [29]:
retriever.display_results(results)

Result 1
Similarity Distance: 0.4900
Metadata: {'moddate': '2026-05-23T00:57:23+00:00', 'page': 5, 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36', 'doc_index': 146, 'creationdate': '2026-05-23T00:57:23+00:00', 'keywords': '', 'source': '..\\data\\pdf_files\\Introduction to AI file (1).pdf', 'modDate': "D:20260523005723+00'00'", 'trapped': '', 'creationDate': "D:20260523005723+00'00'", 'format': 'PDF 1.4', 'producer': 'Skia/PDF m148', 'file_path': '..\\data\\pdf_files\\Introduction to AI file (1).pdf', 'content_length': 967, 'total_pages': 13, 'subject': '', 'title': 'Introduction to AI', 'author': ''}
--------------------------------------------------------------------------------
Examples students see every day
Example
What AI does (simple)
ChatGPT / Gemini
Writes answers, helps homework, explains topics
YouTube / TikTok
Recommends videos you might watch
Google Translate
Translates languages
Face unlock on
p

## Integration RAG pipeline using LLM output

In [30]:
import sys
print(sys.executable)

c:\Users\Hp\Documents\RAG-for begineers\.venv\Scripts\python.exe


In [31]:
import sys
import subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "langchain-groq"])

0

In [3]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()   # <-- Don't forget the parentheses

groq_api_key = os.getenv("Add your GROQ_API_KEY")


llM= ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-120b",temperature=0.1,max_tokens=1024)

def rag_simple(query, retriever, llm):

    # Retrieve relevant documents
    docs = retriever.invoke(query)

    if not docs:
        return "No relevant context found."

    # Combine retrieved text
    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""
You are a helpful AI assistant.

Use ONLY the context below to answer the question.

Context:
{context}

Question:
{query}

Answer:
"""

    response = llm.invoke(prompt)

    return response.content

c:\Users\Hp\Documents\RAG-for begineers\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [19]:
query = "What is Machine Learning? Tell the types of machine learning?"

docs = vectorstore.retrieve(
    query=query,
    embedding_manager=embedding_manager,
    top_k=3
)

print(docs)

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 23.67it/s]

Generated embeddings with shape: (1, 384)
[{'content': 'machine learning basics\n\nMachine learning is a type of artificial intelligence where computers learn from data instead of using rigid rules written by humans. The three core approaches are supervised learning, unsupervised learning, and reinforcement learning\nCore Types of Machine Learning\nSupervised Learning: Trains on labeled data (input and correct output provided) to predict values or categories, such as linear regression or spam filters.\nUnsupervised Learning: Finds hidden patterns or groups in unlabeled data, such as customer clustering or market segmentation.', 'metadata': {'source': '..\\data\\text_files\\machine learning_intro.txt', 'doc_index': 0, 'content_length': 572}, 'distance': 0.48856931924819946}, {'content': 'machine learning basics\n\nMachine learning is a type of artificial intelligence where computers learn from data instead of using rigid rules written by humans. The three core approaches are supervised 